# Kinematic analysis of the skin-stretch 2AFC experiment

Standalone analysis folder. It reads only raw experiment result files (`answers.csv`, `pair_*/tracking.csv`, optional `side_camera.mp4`) and writes all derived outputs to `analysis/Kinematics/results`. It does **not** modify the experiment architecture or raw data.

Main outputs: XY location, orientation, velocity, acceleration, radial/tangential movement, LPF/HPF traces, direction-vs-success summaries, within-subject motor-control normalization, within-finger stiffness effects, paired finger comparisons, polar radiation patterns, distance-from-center accuracy, and side-camera Z/lift proxy by time/stiffness.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display

CWD = Path.cwd()
if (CWD / "kinematics_analysis.py").exists():
    ANALYSIS_DIR = CWD
elif (CWD / "analysis" / "Kinematics" / "kinematics_analysis.py").exists():
    ANALYSIS_DIR = CWD / "analysis" / "Kinematics"
else:
    raise FileNotFoundError("Open this notebook from the project root or analysis/Kinematics")
sys.path.insert(0, str(ANALYSIS_DIR))
import kinematics_analysis as ka

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

DATA_ROOT = Path(r"C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\expiriments_results")
OUTPUT_ROOT = ANALYSIS_DIR / "results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CENTER_X = 320.0
CENTER_Y = 240.0
LOWPASS_CUTOFF_HZ = 3.0
HIGHPASS_CUTOFF_HZ = 0.10
TRAJECTORY_TIME_BINS = 50
SIDE_VIDEO_SAMPLES_PER_TRIAL = 4  # fast estimate across all side videos; increase for denser Z-over-time curves
MAX_TRACKING_TRIALS = None        # set a number for debugging only
MAX_SIDE_VIDEO_TRIALS = None      # set a number for debugging only
FIG_DPI = 160

print("ANALYSIS_DIR", ANALYSIS_DIR.resolve())
print("DATA_ROOT", DATA_ROOT)
print("OUTPUT_ROOT", OUTPUT_ROOT.resolve())

ANALYSIS_DIR C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics
DATA_ROOT C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\expiriments_results
OUTPUT_ROOT C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\results


## 1. Discover trials

Each `answers.csv` row is mapped to `pair_NNN/tracking.csv` and `side_camera.mp4`. P folders are pilots; E folders are experiment participants.

In [2]:
trial_file_manifest = ka.discover_trials(DATA_ROOT)
ka.save_csv(trial_file_manifest, OUTPUT_ROOT, "trial_file_manifest.csv")
print("Discovered rows:", trial_file_manifest.shape)
display(trial_file_manifest.groupby(["subject_group", "selected", "tracking_exists", "side_video_exists"], dropna=False).size().reset_index(name="n"))
display(trial_file_manifest.head())

Discovered rows: (3464, 17)


,subject_group,selected,tracking_exists,side_video_exists,n
0,E,True,False,False,12
1,E,True,True,True,3192
2,P,False,NaN,NaN,2
3,P,True,False,False,1
4,P,True,True,True,256
5,ט,False,NaN,NaN,1


,subject_id,subject_group,selected,source_file,run_dir,trial_index_raw,pair_dir,tracking_file,side_video_file,top_video_file,tracking_exists,side_video_exists,comparison_value,standard_value,signed_stiffness_delta,correct_response,warning
0,E1,E,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,True,True,NaN,NaN,NaN,NaN,
1,E1,E,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,2.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,True,True,1.0,85.0,-84.0,1.0,
2,E1,E,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,3.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,True,True,2.0,85.0,-83.0,1.0,
3,E1,E,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,4.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,True,True,3.0,85.0,-82.0,0.0,
4,E1,E,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,5.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,True,True,4.0,85.0,-81.0,1.0,


## 2. XY kinematics from tracking.csv

For every tracking sample the notebook computes location relative to center, polar orientation, LPF/HPF position components, velocity, acceleration, movement direction, radial velocity, and tangential velocity.

In [3]:
kin = ka.compute_tracking_kinematics(
    trial_file_manifest,
    center_x=CENTER_X,
    center_y=CENTER_Y,
    lowpass_cutoff_hz=LOWPASS_CUTOFF_HZ,
    highpass_cutoff_hz=HIGHPASS_CUTOFF_HZ,
    n_time_bins=TRAJECTORY_TIME_BINS,
    max_trials=MAX_TRACKING_TRIALS,
)
kinematic_samples = kin["samples"]
trial_kinematic_summary = kin["trial_summary"]          # one row per participant x pair x stiffness segment
pair_kinematic_summary = kin["pair_summary"]            # one row per recorded pair
trajectory_time_bins = kin["time_bins"]                  # time normalized within each stiffness segment

ka.save_csv(kinematic_samples, OUTPUT_ROOT, "kinematic_samples.csv")
ka.save_csv(pair_kinematic_summary, OUTPUT_ROOT, "pair_kinematic_summary.csv")
ka.save_csv(trial_kinematic_summary, OUTPUT_ROOT, "trial_kinematic_summary.csv")
ka.save_csv(trajectory_time_bins, OUTPUT_ROOT, "trajectory_time_bins.csv")
print("samples", kinematic_samples.shape, "stiffness trials", trial_kinematic_summary.shape, "pairs", pair_kinematic_summary.shape, "time bins", trajectory_time_bins.shape)
display(trial_kinematic_summary.head())


C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\kinematics_analysis.py:212: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d["timestamp_dt"] = pd.to_datetime(d["timestamp"], errors="coerce")
C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\kinematics_analysis.py:212: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d["timestamp_dt"] = pd.to_datetime(d["timestamp"], errors="coerce")
C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\kinematics_analysis.py:212: UserWarning: Could not infer format, so each element will be parsed i

C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\kinematics_analysis.py:212: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d["timestamp_dt"] = pd.to_datetime(d["timestamp"], errors="coerce")
C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\kinematics_analysis.py:212: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  d["timestamp_dt"] = pd.to_datetime(d["timestamp"], errors="coerce")
C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\kinematics_analysis.py:212: UserWarning: Could not infer format, so each element will be parsed i

samples (1445602, 30) trials (3451, 39) time bins (172386, 28)


,subject_id,subject_group,selected,source_file,run_dir,trial_index_raw,pair_dir,tracking_file,side_video_file,top_video_file,tracking_exists,side_video_exists,comparison_value,standard_value,signed_stiffness_delta,correct_response,warning,finger_condition,tracking_warning,n_tracking_samples,duration_s,sampling_rate_hz,mean_r_center_px,max_r_center_px,path_length_px,net_displacement_px,straightness_index,mean_speed_px_s,max_speed_px_s,mean_acceleration_px_s2,max_acceleration_px_s2,mean_radial_velocity_px_s,mean_abs_radial_velocity_px_s,mean_abs_tangential_velocity_px_s,dominant_movement_angle_deg,dominant_movement_direction,movement_direction_resultant_length,movement_direction_entropy_bits,mean_skin_stretch_gain_mm_per_m
0,E1,E,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,True,True,NaN,NaN,NaN,NaN,,pinky,,1146.0,52.141410,21.172535,114.063903,240.497221,3040.848411,6.797698,0.002235,121.917155,437.229903,466.419214,2822.184352,1.163319,103.712879,44.704322,21.927596,SE,0.009948,3.493922,85.0
1,E1,E,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,2.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,True,True,1.0,85.0,-84.0,1.0,,pinky,,520.0,25.755411,21.068155,135.108517,228.667367,4049.437745,10.763515,0.002658,163.034004,782.311715,629.593102,9627.768056,7.549189,124.321287,71.292473,8.759751,SE,0.032223,3.421895,45.0
2,E1,E,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,3.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,True,True,2.0,85.0,-83.0,1.0,,pinky,,559.0,28.210318,20.985698,155.545087,257.149425,5179.703997,0.000000,0.000000,193.437796,648.052149,750.033029,3522.444325,1.509176,103.222110,131.065975,-56.349548,S,0.001200,3.479478,25.0
3,E1,E,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,4.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,True,True,3.0,85.0,-82.0,0.0,,pinky,,914.0,45.811366,21.038458,140.207435,253.636417,7180.525022,168.560913,0.023475,160.374189,647.965154,616.831789,8122.721142,7.173199,93.079751,106.336945,86.298151,E,0.038326,3.438474,165.0
4,E1,E,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,5.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,True,True,4.0,85.0,-81.0,1.0,,pinky,,466.0,23.140600,21.102389,119.693062,221.612428,3965.088176,7.051042,0.001778,184.575931,503.293356,698.379076,4344.128414,5.648825,95.851179,127.330199,-77.950374,SSW,0.027580,3.439835,85.0


## 3. Direction, distance, and success summaries

These tables test whether movement direction, distance from center, speed, or acceleration are associated with selection accuracy.

In [4]:
summaries = ka.summarize_kinematics(trial_kinematic_summary, trajectory_time_bins)
direction_success_summary = summaries["direction_success"]
distance_success_summary = summaries["distance_success"]
subject_kinematic_summary = summaries["subject_summary"]
participant_stiffness_kinematic_summary = summaries["participant_stiffness_summary"]
stiffness_kinematic_summary = summaries["stiffness_summary"]
group_time_summary = summaries["group_time"]
stiffness_time_summary = summaries["stiffness_time"]
kinematic_group_metric_summary = summaries["kinematic_group_metric_summary"]
kinematic_group_condition_metric_summary = summaries["kinematic_group_condition_metric_summary"]
kinematic_within_group_condition_comparisons = summaries["kinematic_within_group_condition_comparisons"]
kinematic_between_group_metric_comparisons = summaries["kinematic_between_group_metric_comparisons"]
kinematic_expanded_scope_metric_summary = summaries.get("kinematic_expanded_scope_metric_summary", pd.DataFrame())
kinematic_expanded_scope_pairwise_mean_differences = summaries.get("kinematic_expanded_scope_pairwise_mean_differences", pd.DataFrame())
kinematic_expanded_scope_status = summaries.get("kinematic_expanded_scope_status", pd.DataFrame())

ka.save_csv(direction_success_summary, OUTPUT_ROOT, "direction_success_summary.csv")
ka.save_csv(distance_success_summary, OUTPUT_ROOT, "distance_success_summary.csv")
ka.save_csv(subject_kinematic_summary, OUTPUT_ROOT, "subject_kinematic_summary.csv")
ka.save_csv(participant_stiffness_kinematic_summary, OUTPUT_ROOT, "participant_stiffness_kinematic_summary.csv")
ka.save_csv(stiffness_kinematic_summary, OUTPUT_ROOT, "stiffness_kinematic_summary.csv")
ka.save_csv(group_time_summary, OUTPUT_ROOT, "group_time_summary.csv")
ka.save_csv(stiffness_time_summary, OUTPUT_ROOT, "stiffness_time_summary.csv")
ka.save_csv(kinematic_group_metric_summary, OUTPUT_ROOT, "kinematic_group_metric_summary.csv")
ka.save_csv(kinematic_group_condition_metric_summary, OUTPUT_ROOT, "kinematic_group_condition_metric_summary.csv")
ka.save_csv(kinematic_within_group_condition_comparisons, OUTPUT_ROOT, "kinematic_within_group_condition_comparisons.csv")
ka.save_csv(kinematic_between_group_metric_comparisons, OUTPUT_ROOT, "kinematic_between_group_metric_comparisons.csv")
ka.save_csv(kinematic_expanded_scope_metric_summary, OUTPUT_ROOT, "kinematic_expanded_scope_metric_summary.csv")
ka.save_csv(kinematic_expanded_scope_pairwise_mean_differences, OUTPUT_ROOT, "kinematic_expanded_scope_pairwise_mean_differences.csv")
ka.save_csv(kinematic_expanded_scope_status, OUTPUT_ROOT, "kinematic_expanded_scope_status.csv")

print("Direction x success per participant/stiffness")
display(direction_success_summary.sort_values(["subject_id", "stiffness_value", "finger_condition", "dominant_movement_direction"]).head(40))
print("Participant x stiffness kinematic summary")
display(participant_stiffness_kinematic_summary.head(40))
print("All participants by stiffness")
display(stiffness_kinematic_summary.sort_values("stiffness_value").head(40))

print("N_E / L_E / L_P kinematic group summary")
display(kinematic_group_metric_summary.head(80))
print("Between-group kinematic comparisons")
display(kinematic_between_group_metric_comparisons.head(80))
print("Expanded requested scopes: all, protocol, E/P, N/L, sex, age, stiffness, finger, success/failure")
display(kinematic_expanded_scope_status)
display(kinematic_expanded_scope_metric_summary.head(120))


Direction x success


,subject_group,finger_condition,dominant_movement_direction,n_trials,success_rate,mean_speed_px_s,mean_r_center_px,mean_path_length_px
0,E,index,E,25,0.375000,174.909543,74.617618,2502.708760
1,E,index,ENE,36,0.571429,201.740313,75.068329,1759.746641
2,E,index,ESE,26,0.423077,198.042499,70.749693,1732.100134
3,E,index,NE,63,0.564516,244.510207,74.526862,2462.760914
4,E,index,NNE,89,0.483146,222.121106,77.837364,1829.643073
5,E,index,S,80,0.384615,209.450633,79.628223,2111.915302
6,E,index,SE,73,0.547945,169.632735,76.165664,1676.928277
7,E,index,SSE,132,0.568182,201.577159,76.643739,1596.370790
8,E,index,SSW,73,0.513889,214.283489,79.285517,2095.453459
9,E,index,SW,54,0.557692,199.283688,72.489195,1878.964405


Subject/finger kinematic summary


,subject_id,subject_group,finger_condition,n_trials,success_rate,mean_max_r_center_px,mean_speed_px_s,mean_acceleration_px_s2,mean_path_length_px,mean_straightness_index,circular_mean_direction_deg,movement_direction_resultant_length
0,E1,E,index,64,0.507937,185.703294,244.786903,1444.959443,1998.735508,0.012354,-80.517580,0.292202
1,E1,E,middle,64,0.468750,186.817727,237.990337,1530.499885,1806.363487,0.008251,-73.923695,0.250319
2,E1,E,pinky,64,0.523810,241.097027,282.481113,1358.960185,4051.886667,0.003480,-84.810408,0.243984
3,E1,E,ring,64,0.468750,177.589459,242.702392,1518.778083,1506.215390,0.010708,-76.702225,0.268362
4,E10,E,index,67,0.393939,144.925131,136.074616,867.653732,902.662065,0.003748,-9.818276,0.631463
5,E10,E,middle,67,0.432836,161.492908,124.659896,832.646892,883.848786,0.005282,-20.397961,0.468541
6,E10,E,pinky,67,0.454545,152.727552,112.274787,899.643765,1088.806401,0.007591,-11.634041,0.524310
7,E10,E,ring,67,0.462687,197.903605,149.320269,1062.806249,2808.797176,0.004023,-18.727806,0.327005
8,E11,E,index,67,0.500000,201.387539,249.794646,1496.075578,2287.854174,0.001630,-51.658233,0.162902
9,E11,E,middle,67,0.552239,190.434790,135.002312,750.592887,1334.822285,0.001848,1.216025,0.046901


## 4. Motor-control repeated-measures calculations

These calculations avoid trial-pooling across people. They produce: within-subject centered/z-scored kinematics, within-finger stiffness sensitivities for each subject, and paired finger comparisons using only subjects who contributed both fingers in a contrast. This is the preferred structure for motor-control interpretation.


In [ ]:
motor_control = ka.compute_motor_control_comparisons(subject_kinematic_summary)
kinematic_within_subject = motor_control["within_subject"]
finger_metric_summary = motor_control["finger_metric_summary"]
within_finger_stiffness_effects = motor_control["within_finger_stiffness_effects"]
within_finger_stiffness_effect_summary = motor_control["within_finger_stiffness_effect_summary"]
finger_comparison_paired = motor_control["finger_comparison_paired"]
finger_comparison_by_stiffness_paired = motor_control["finger_comparison_by_stiffness_paired"]

ka.save_csv(kinematic_within_subject, OUTPUT_ROOT, "kinematic_within_subject.csv")
ka.save_csv(finger_metric_summary, OUTPUT_ROOT, "finger_metric_summary.csv")
ka.save_csv(within_finger_stiffness_effects, OUTPUT_ROOT, "within_finger_stiffness_effects.csv")
ka.save_csv(within_finger_stiffness_effect_summary, OUTPUT_ROOT, "within_finger_stiffness_effect_summary.csv")
ka.save_csv(finger_comparison_paired, OUTPUT_ROOT, "finger_comparison_paired.csv")
ka.save_csv(finger_comparison_by_stiffness_paired, OUTPUT_ROOT, "finger_comparison_by_stiffness_paired.csv")

motor_control_figure_paths = ka.save_motor_control_figures(OUTPUT_ROOT, motor_control, fig_dpi=FIG_DPI)

print("Within-subject rows:", kinematic_within_subject.shape)
display(kinematic_within_subject.head(30))
print("Within-finger stiffness effects: one slope per subject x finger x metric")
display(within_finger_stiffness_effects.head(30))
print("Within-finger stiffness-effect summary")
display(within_finger_stiffness_effect_summary.sort_values(["metric", "finger_condition"]).head(80))
print("Paired finger comparisons (finger_b - finger_a)")
display(finger_comparison_paired.sort_values(["metric", "comparison"]).head(80))
print(f"Saved {len(motor_control_figure_paths)} motor-control figures")


## 5. Side-camera Z/lift estimate

Current experiment code records side-camera video but does not save a direct Z column in `tracking.csv`. This section estimates a **pixel Z/lift proxy** from `side_camera.mp4` using a fast skin-color mask. Positive values mean the hand/finger region is higher in the side image relative to the within-trial baseline. This is an estimate, not calibrated millimetres.

In [5]:
side = ka.estimate_side_video_z(
    trial_kinematic_summary,
    samples_per_video=SIDE_VIDEO_SAMPLES_PER_TRIAL,
    max_trials=MAX_SIDE_VIDEO_TRIALS,
)
side_z_samples = side["side_samples"]
side_z_trial_summary = side["side_trial_summary"]
side_z_group_time_summary = side["side_group_time"]
side_z_subject_stiffness_summary = side["side_subject_stiffness_summary"]
side_z_by_stiffness_summary = side["side_stiffness_summary"]

ka.save_csv(side_z_samples, OUTPUT_ROOT, "side_z_samples.csv")
ka.save_csv(side_z_trial_summary, OUTPUT_ROOT, "side_z_trial_summary.csv")
ka.save_csv(side_z_group_time_summary, OUTPUT_ROOT, "side_z_group_time_summary.csv")
ka.save_csv(side_z_subject_stiffness_summary, OUTPUT_ROOT, "side_z_subject_stiffness_summary.csv")
ka.save_csv(side_z_by_stiffness_summary, OUTPUT_ROOT, "side_z_by_stiffness_summary.csv")

print("side samples", side_z_samples.shape, "side stiffness trial summary", side_z_trial_summary.shape)
display(side_z_trial_summary.head())
print("Z/lift by stiffness")
display(side_z_by_stiffness_summary.sort_values("stiffness_value").head(30))


side samples (13786, 54) side trial summary (3451, 14)


,subject_id,trial_index_raw,subject_group,finger_condition,comparison_value,standard_value,signed_stiffness_delta,correct_response,side_video_file,n_side_samples,side_detection_rate,mean_side_z_lift_px,max_side_z_lift_px,mean_side_mask_area_px
0,E1,1.0,E,pinky,NaN,NaN,NaN,NaN,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,4,1.0,61.90,164.4000,33395.25
1,E1,2.0,E,pinky,1.0,85.0,-84.0,1.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,4,1.0,53.63,107.8925,22666.25
2,E1,3.0,E,pinky,2.0,85.0,-83.0,1.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,4,1.0,63.20,199.9500,35106.00
3,E1,4.0,E,pinky,3.0,85.0,-82.0,0.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,4,1.0,43.25,109.0000,19326.50
4,E1,5.0,E,pinky,4.0,85.0,-81.0,1.0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,4,1.0,29.05,64.8000,38061.50


Z/lift by stiffness


,subject_group,comparison_value,n_trials,mean_side_z_lift_px,sem_side_z_lift_px,max_side_z_lift_px,success_rate
0,E,1.0,12,49.976667,8.069099,113.022500,0.750000
1,E,2.0,12,40.557500,7.943050,86.445000,0.416667
2,E,3.0,12,42.411111,9.948276,100.029167,0.333333
3,E,4.0,12,42.435486,7.685117,88.785833,0.750000
4,E,5.0,12,33.995833,6.204765,69.370833,0.500000
5,E,6.0,12,33.137500,7.278260,74.016667,0.333333
6,E,7.0,12,43.511667,9.730481,91.703333,0.250000
7,E,8.0,12,58.541667,13.927108,117.770833,0.833333
8,E,9.0,12,49.340278,7.278676,100.270833,0.500000
9,E,10.0,12,43.006250,10.244470,89.470833,0.250000


## 6. Advanced success, Z-height, and trajectory structure

IEEE-style motor-control layer: successful-vs-unsuccessful kinematic/Z contrasts are computed within subject and finger; trajectory templates are compared between fingers and between correct/incorrect trials using normalized XY paths.


In [ ]:
success_kinematic_z = ka.compute_success_kinematic_z_analysis(trial_kinematic_summary, side_z_trial_summary)
trial_success_kinematic_z_table = success_kinematic_z["trial_success_table"]
success_kinematic_z_contrast_by_subject_finger = success_kinematic_z["success_contrast_by_subject_finger"]
success_kinematic_z_contrast_summary = success_kinematic_z["success_contrast_summary"]
success_kinematic_z_contrast_by_finger_summary = success_kinematic_z["success_contrast_by_finger_summary"]

trajectory_structure = ka.compute_trajectory_similarity_analysis(trajectory_time_bins)
subject_finger_trajectory = trajectory_structure["subject_finger_trajectory"]
trajectory_variability_summary = trajectory_structure["trajectory_variability_summary"]
finger_trajectory_distance_paired = trajectory_structure["finger_trajectory_distance_paired"]
finger_trajectory_distance_summary = trajectory_structure["finger_trajectory_distance_summary"]
success_failure_trajectory_distance = trajectory_structure["success_failure_trajectory_distance"]
success_failure_trajectory_distance_summary = trajectory_structure["success_failure_trajectory_distance_summary"]

ka.save_csv(trial_success_kinematic_z_table, OUTPUT_ROOT, "trial_success_kinematic_z_table.csv")
ka.save_csv(success_kinematic_z_contrast_by_subject_finger, OUTPUT_ROOT, "success_kinematic_z_contrast_by_subject_finger.csv")
ka.save_csv(success_kinematic_z_contrast_summary, OUTPUT_ROOT, "success_kinematic_z_contrast_summary.csv")
ka.save_csv(success_kinematic_z_contrast_by_finger_summary, OUTPUT_ROOT, "success_kinematic_z_contrast_by_finger_summary.csv")
ka.save_csv(subject_finger_trajectory, OUTPUT_ROOT, "subject_finger_trajectory.csv")
ka.save_csv(trajectory_variability_summary, OUTPUT_ROOT, "trajectory_variability_summary.csv")
ka.save_csv(finger_trajectory_distance_paired, OUTPUT_ROOT, "finger_trajectory_distance_paired.csv")
ka.save_csv(finger_trajectory_distance_summary, OUTPUT_ROOT, "finger_trajectory_distance_summary.csv")
ka.save_csv(success_failure_trajectory_distance, OUTPUT_ROOT, "success_failure_trajectory_distance.csv")
ka.save_csv(success_failure_trajectory_distance_summary, OUTPUT_ROOT, "success_failure_trajectory_distance_summary.csv")

advanced_figure_paths = ka.save_advanced_kinematic_figures(OUTPUT_ROOT, success_kinematic_z, trajectory_structure, fig_dpi=FIG_DPI)

print("Success-linked kinematic/Z contrasts")
display(success_kinematic_z_contrast_summary.sort_values("metric").head(80))
print("Between-finger trajectory distance summary")
display(finger_trajectory_distance_summary.head(80))
print("Correct-vs-incorrect trajectory distance summary")
display(success_failure_trajectory_distance_summary.head(80))
print(f"Saved {len(advanced_figure_paths)} advanced figures")


## 7. Subject-specific XY trajectories in space

This is the subject-first view: one spatial XY trajectory table and one figure per subject. No pooling across participants is used in these outputs. Thin lines show that subject's stiffness-specific trajectories; thick lines show that same subject's finger-average trajectory.


In [ ]:
subject_spatial = ka.compute_subject_spatial_trajectory_analysis(trajectory_time_bins)
subject_xy_trajectory = subject_spatial["subject_xy_trajectory"]
subject_spatial_trajectory_summary = subject_spatial["subject_spatial_trajectory_summary"]
subject_finger_spatial_distance = subject_spatial["subject_finger_spatial_distance"]
subject_spatial_metric_distribution = subject_spatial["subject_spatial_metric_distribution"]

ka.save_csv(subject_xy_trajectory, OUTPUT_ROOT, "subject_xy_trajectory.csv")
ka.save_csv(subject_spatial_trajectory_summary, OUTPUT_ROOT, "subject_spatial_trajectory_summary.csv")
ka.save_csv(subject_finger_spatial_distance, OUTPUT_ROOT, "subject_finger_spatial_distance.csv")
ka.save_csv(subject_spatial_metric_distribution, OUTPUT_ROOT, "subject_spatial_metric_distribution.csv")
subject_xy_figure_paths = ka.save_subject_xy_trajectory_figures(OUTPUT_ROOT, subject_xy_trajectory, fig_dpi=FIG_DPI)

print("Subject XY trajectory table:", subject_xy_trajectory.shape)
display(subject_xy_trajectory.head(30))
print("Subject-specific spatial trajectory summary:")
display(subject_spatial_trajectory_summary.sort_values(["subject_id", "finger_condition", "stiffness_value"]).head(80))
print("Subject-specific finger spatial distances:")
display(subject_finger_spatial_distance.sort_values(["subject_id", "stiffness_value", "comparison"]).head(80))
print("Subject spatial metric distributions with mean/median/95% CI and log-backtransformed summaries:")
display(subject_spatial_metric_distribution.head(80))
print(f"Saved {len(subject_xy_figure_paths)} per-subject XY figures")


## 8. Subject-specific velocity and acceleration

Same subject-first structure as the XY spatial plots, now for velocity and acceleration. The profiles use derivatives computed inside each stiffness segment, so velocity/acceleration do not bridge object transitions.


In [ ]:
subject_va = ka.compute_subject_velocity_acceleration_analysis(trajectory_time_bins)
subject_velocity_acceleration_profile = subject_va["subject_velocity_acceleration_profile"]
subject_velocity_acceleration_summary = subject_va["subject_velocity_acceleration_summary"]
subject_finger_velocity_acceleration_distance = subject_va["subject_finger_velocity_acceleration_distance"]
subject_velocity_acceleration_metric_distribution = subject_va["subject_velocity_acceleration_metric_distribution"]
velocity_stiffness_influence_summary = subject_va.get("velocity_stiffness_influence_summary", pd.DataFrame())
velocity_finger_influence_summary = subject_va.get("velocity_finger_influence_summary", pd.DataFrame())
velocity_time_influence_summary = subject_va.get("velocity_time_influence_summary", pd.DataFrame())

ka.save_csv(subject_velocity_acceleration_profile, OUTPUT_ROOT, "subject_velocity_acceleration_profile.csv")
ka.save_csv(subject_velocity_acceleration_summary, OUTPUT_ROOT, "subject_velocity_acceleration_summary.csv")
ka.save_csv(subject_finger_velocity_acceleration_distance, OUTPUT_ROOT, "subject_finger_velocity_acceleration_distance.csv")
ka.save_csv(subject_velocity_acceleration_metric_distribution, OUTPUT_ROOT, "subject_velocity_acceleration_metric_distribution.csv")
ka.save_csv(velocity_stiffness_influence_summary, OUTPUT_ROOT, "velocity_stiffness_influence_summary.csv")
ka.save_csv(velocity_finger_influence_summary, OUTPUT_ROOT, "velocity_finger_influence_summary.csv")
ka.save_csv(velocity_time_influence_summary, OUTPUT_ROOT, "velocity_time_influence_summary.csv")
subject_va_figure_paths = ka.save_subject_velocity_acceleration_figures(OUTPUT_ROOT, subject_velocity_acceleration_profile, fig_dpi=FIG_DPI)

print("Subject velocity/acceleration profile:", subject_velocity_acceleration_profile.shape)
display(subject_velocity_acceleration_profile.head(30))
print("Subject velocity/acceleration summary:")
display(subject_velocity_acceleration_summary.sort_values(["subject_id", "finger_condition", "stiffness_value"]).head(80))
print("Subject-specific finger velocity/acceleration distances:")
display(subject_finger_velocity_acceleration_distance.sort_values(["subject_id", "stiffness_value", "comparison"]).head(80))
print("Subject velocity/acceleration metric distributions with mean/median/95% CI and log-backtransformed summaries:")
display(subject_velocity_acceleration_metric_distribution.head(80))
print("Velocity influence answers: stiffness, fingers, and time thirds")
display(velocity_stiffness_influence_summary)
display(velocity_finger_influence_summary)
display(velocity_time_influence_summary.head(80))
print(f"Saved {len(subject_va_figure_paths)} per-subject velocity/acceleration figures")


## 9. 3D proxy kinematics, confidence intervals, and optional LMM

Combines top-camera XY with side-camera Z/lift proxy inside each subject/trial/stiffness segment. Outputs include 3D path length, max excursion, peak velocity, peak acceleration, median and 95% confidence intervals, log-domain summaries back-transformed to original units, individual data points in the direction/stiffness figure, and an optional mixed-effects model if `statsmodels` is installed.


In [ ]:
proxy3d = ka.compute_3d_proxy_kinematics(kinematic_samples, side_z_samples)
kinematic_3d_proxy_samples = proxy3d["kinematic_3d_proxy_samples"]
trial_3d_kinematic_summary = proxy3d["trial_3d_kinematic_summary"]
subject_3d_kinematic_summary = proxy3d["subject_3d_kinematic_summary"]
subject_3d_metric_distribution = proxy3d["subject_3d_metric_distribution"]
stiffness_direction_3d_metric_distribution = proxy3d["stiffness_direction_3d_metric_distribution"]
mixedlm_model_input = proxy3d["mixedlm_model_input"]

ka.save_csv(kinematic_3d_proxy_samples, OUTPUT_ROOT, "kinematic_3d_proxy_samples.csv")
ka.save_csv(trial_3d_kinematic_summary, OUTPUT_ROOT, "trial_3d_kinematic_summary.csv")
ka.save_csv(subject_3d_kinematic_summary, OUTPUT_ROOT, "subject_3d_kinematic_summary.csv")
ka.save_csv(subject_3d_metric_distribution, OUTPUT_ROOT, "subject_3d_metric_distribution.csv")
ka.save_csv(stiffness_direction_3d_metric_distribution, OUTPUT_ROOT, "stiffness_direction_3d_metric_distribution.csv")
ka.save_csv(mixedlm_model_input, OUTPUT_ROOT, "mixedlm_model_input.csv")

mixedlm = ka.fit_optional_mixed_effects_models(mixedlm_model_input)
ka.save_csv(mixedlm["mixedlm_status"], OUTPUT_ROOT, "mixedlm_status.csv")
ka.save_csv(mixedlm["mixedlm_coefficients"], OUTPUT_ROOT, "mixedlm_coefficients.csv")
proxy_3d_figure_paths = ka.save_3d_proxy_figures(OUTPUT_ROOT, kinematic_3d_proxy_samples, trial_3d_kinematic_summary, fig_dpi=FIG_DPI)

print("3D proxy trial summary:", trial_3d_kinematic_summary.shape)
display(trial_3d_kinematic_summary.head(30))
print("3D subject summaries:")
display(subject_3d_kinematic_summary.head(30))
print("3D metric distributions: mean, median, 95% CI, log-transform recommendation, and back-transformed geometric mean/CI")
display(subject_3d_metric_distribution.head(80))
print("MixedLM status / coefficients")
display(mixedlm["mixedlm_status"])
display(mixedlm["mixedlm_coefficients"].head(80))
print(f"Saved {len(proxy_3d_figure_paths)} 3D proxy figures")


## 10. Hand / active-finger orientation planes

Summarizes the thumb-to-active-finger orientation in XY from the top camera and YZ/ZX from the side-camera Z/lift proxy. The figures include three side-by-side vector panels (XY, YZ, ZX): faded vectors are individual trials and thick colored vectors are group means, matching common motor-control vector-field / mean-vector visualization practice. Angles stay in camera/proxy units while workspace labels document the L=60x60 cm and N=40x50 cm normalization context.


In [ ]:
hand_orientation = ka.compute_hand_orientation_plane_analysis(trial_kinematic_summary, side_z_trial_summary)
hand_orientation_plane_trials = hand_orientation["hand_orientation_plane_trials"]
hand_orientation_plane_summary = hand_orientation["hand_orientation_plane_summary"]

ka.save_csv(hand_orientation_plane_trials, OUTPUT_ROOT, "hand_orientation_plane_trials.csv")
ka.save_csv(hand_orientation_plane_summary, OUTPUT_ROOT, "hand_orientation_plane_summary.csv")
hand_orientation_figure_paths = ka.save_hand_orientation_plane_figures(
    OUTPUT_ROOT, hand_orientation_plane_summary, hand_orientation_plane_trials, fig_dpi=FIG_DPI
)

print("Hand orientation plane trials:", hand_orientation_plane_trials.shape)
display(hand_orientation_plane_summary.head(30))
print(f"Saved {len(hand_orientation_figure_paths)} hand-orientation figures")


## 11. Figures

All figures are saved under `analysis/Kinematics/results/figures`, including polar radiation patterns per participant and for all/E/P, side-by-side success/failure polar dwell-time plots with semi-transparent stiffness colors, and side-Z participant plots over time and over stiffness. Group summary graphs include faded raw values behind the summary bars when raw values are available.


In [6]:
figure_paths = ka.save_kinematic_figures(
    OUTPUT_ROOT,
    trial_kinematic_summary,
    group_time_summary,
    direction_success_summary,
    distance_success_summary,
    side_z_group_time_summary,
    side_z_by_stiffness_summary,
    fig_dpi=FIG_DPI,
)
print(f"Saved {len(figure_paths)} figures")
display(pd.DataFrame({"figure": [str(p) for p in figure_paths]}).head(80))

Saved 26 figures


,figure
0,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
1,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
2,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
3,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
4,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
5,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
6,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
7,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
8,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
9,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...


## 12. Manifest and interpretation notes

- `kinematic_samples.csv`: sample-level XY time series with LPF/HPF, velocity, acceleration, and direction.
- `trial_kinematic_summary.csv`: per-trial features for modeling success.
- `direction_success_summary.csv`: direction bins vs success.
- `distance_success_summary.csv`: distance-from-center vs success.
- `kinematic_within_subject.csv`: subject-preserving table with centered/z-scored kinematic metrics.
- `within_finger_stiffness_effects.csv`: per-subject/per-finger stiffness slopes and high-minus-low deltas.
- `finger_comparison_paired.csv`: paired within-subject finger contrasts.
- `success_kinematic_z_*`: within-subject/finger successful-vs-unsuccessful contrasts including side-camera Z/lift.
- `trajectory_*`: normalized trajectory variability, paired between-finger trajectory distances, and correct-vs-incorrect trajectory distances.
- `subject_xy_trajectory.csv`, `subject_spatial_trajectory_summary.csv`, and `subject_finger_spatial_distance.csv`: strictly per-subject spatial XY trajectory outputs.
- `figures/subject_xy_trajectories/`: one XY-in-space figure per subject.
- `subject_velocity_acceleration_profile.csv`, `subject_velocity_acceleration_summary.csv`, and `subject_finger_velocity_acceleration_distance.csv`: strictly per-subject velocity/acceleration outputs.
- `*_metric_distribution.csv`: data-point counts, mean/median, 95% CIs, skewness, log-transform recommendation, and back-transformed geometric mean/CI.
- `trial_3d_kinematic_summary.csv`, `subject_3d_kinematic_summary.csv`: 3D proxy path length, max excursion, peak velocity, peak acceleration, and Z/lift summaries.
- `mixedlm_*`: optional mixed-effects model outputs; if `statsmodels` is absent, `mixedlm_status.csv` explains the skipped model.
- `figures/subject_velocity_acceleration/`: one velocity/acceleration figure per subject.
- `side_z_*`: side-camera Z/lift proxy estimated from video pixels.
- Side-camera Z is an image-based estimate and should be interpreted qualitatively unless you add a physical pixel-to-mm calibration or a manually verified segmentation method.
- `curvature_1_px`, `mean_curvature_1_px`, `speed_curvature_power_law_slope`, and `speed_curvature_power_law_r2` add the literature-guided speed-curvature / two-thirds-power-law check (`speed ? curvature^-1/3`) from motor-control reaching/drawing analyses. These values are included in subject and group summaries when the trajectory has enough valid curved samples.


In [7]:
manifest = ka.analysis_manifest(OUTPUT_ROOT)
display(manifest)
print("Figures:", OUTPUT_ROOT / "figures")

,output,exists,path
0,trial_file_manifest.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
1,kinematic_samples.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
2,trial_kinematic_summary.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
3,trajectory_time_bins.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
4,direction_success_summary.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
5,distance_success_summary.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
6,subject_kinematic_summary.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
7,group_time_summary.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
8,side_z_samples.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...
9,side_z_trial_summary.csv,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...


Figures: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\Kinematics\results\figures
